In [13]:
""" Created on November 13, 2023 // Updated on March 20, 2026 // @author: Sarah Shi """

import os
import numpy as np
import pandas as pd

import sys 
sys.path.append('../../src')
import mineralML as mm

import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

## mineralML Mapping Quickstart

This notebook shows **how to load and run your quantitative EDS data through mineralML** with an example gabbroic nodule from the Galapagos: `09g3`. These data are from Gleeson et al., 2024 (find the paper here: https://doi.org/10.1093/petrology/egaf031) in this GitHub repository (https://github.com/gleesonm1/GleesonEtAl_JPet_2024_supplement/tree/main/Code_Figures/Data/LargeScaleMaps/Gabbro%20Samples). Please refer to the paper for more information about this sample. 

This is a four step process: 
1. Load a directory containing all your CSVs of mapped chemical data with `load_df` (or `pd.read_csv` directly).
2. [Optional] Convert the input from element to oxide wt%.
3. Predict the mineral class with mineralML.
3. Plot the phase map, plot phase_counts, and plot probability histograms

I have conveniently (I hope!) wrapped all of these bits into one function, called ``mm.run_map``.

We loaded in the ``mineralML`` Python package as ``mm``. ``mineralML`` has trained machine learning models for classifying minerals. This implementation aims to get your electron microprobe or quantitative EDS compositions classified and processed. We remove some degrees of freedom to simplify the process as much as possible. The minerals considered for this study include: Amphibole, Apatite, Biotite, Calcite, Chlorite, Epidote, Feldspar (KFeldspar and Plagioclase), Garnet, Glass, Kalsilite, Leucite, Melilite, Muscovite, Nepheline, Olivine, Pyroxene (Clinopyroxene, Orthopyroxene, Sodic Pyroxenes), Quartz, Rhombohedral_Oxides (Hematite-Ilmenite), Rutile, Serpentine, Spinels (Magnetite-Spinel), Titanite, Tourmaline, and Zircon. 

One CSV file containing your electron microprobe analyses in oxide weight percentages is necessary. Find an example [here](https://github.com/sarahshi/mineralML/blob/main/docs/examples/09g3). The nominally necessary oxides are $SiO_2$, $TiO_2$, $Al_2O_3$, $FeO_t$, $MnO$, $MgO$, $CaO$, $Na_2O$, $K_2O$, $Cr_2O_3$, and $P_2O_5$. For the oxides not analyzed for specific minerals, the preprocessing will fill in the nan values as 0. 

## Supervised Machine Learning for SEM-EDS Data

## Load and prepare data for analysis

Here, we will work with data that are in elemental weight percent. This means that we will have to do a conversion to oxide weight perecent.

In [ ]:
# Find your directory of mapped mineral data, stored in Maps/09g3. 
# This code identifies any file with CSV and appends it to the map. 

base = "Maps"
map_dirs = []
for root, subdirs, files in os.walk(base):
    # Skip any path that includes 'Ignore' in its folder names
    if "Ignore" in root.split(os.sep):
        continue
    
    if any(f.lower().endswith(".csv") for f in files):
        map_dirs.append(root)

print(map_dirs)

## mm.mm.run_map

We will use ``mm.mm.run_map`` which will return all you need! 

In [ ]:
# Inspect the inputs and outputs of mm.mm.run_map
help(mm.run_map)

In [ ]:
# Here is our all in one function! Read the inputs and outputs provided above. 
 
output = mm.run_map(map_dirs[0], # provide the directory of interest. alternatively, you can provide the preloaded dictionary of oxides. 
                    prob_threshold=0.6, # provide a probability threshold. here, i only want values with >= 0.6 (60%) probability.
                    min_frac=0.01, # provide a minimum pixel fraction for the phase to be displayed
                    units='element_wt%', # provide the unit. can choose 'element_wt%' or 'oxide_wt%'
                    phases=['Plagioclase', 'Clinopyroxene', 'Orthopyroxene', 'Spinels', 'Olivine', 'Glass'], # phases of interest
                    scalebar_um=50, # define size of scalebar desired, in microns
                    pixel_size_um=2, # define size of each pixel of scalebar, in microns 
                    scalebar_loc='upper right', # specify location for scalebar
                    scalebar_col='white', # specify color for scalebar
                    )


In [ ]:
# Inspect what is in the outputs
output.keys()

Let's say you now want to work with these data in dataframe form rather than dictionary form. How would you do this? 

In [ ]:
# Pull the dataframe of predictions
df_pred = output['df_pred']
display(df_pred)

Let's plot the prediction scores from the output, in mapped form. This allows for further investigation to determine where predictions are more and less certain. 

In [ ]:
fig, ax = mm.plot_score_map(
    output['prob_map'], # take the probability map from the output to plot
    scalebar_um=50, # define size of scalebar desired, in microns
    pixel_size_um=2, # define size of each pixel of scalebar, in microns 
    scalebar_loc='upper right', # specify location for scalebar
    scalebar_col='black', # specify color for scalebar
)

### We can do some more with mineralML now. Let's plot all the feldspars, pyroxenes, and spinels in ternary space. 


In [ ]:
# Here are all our feldspars 
fspars = df_pred[df_pred.Predict_Mineral == 'Plagioclase']
display('Feldspars:', fspars)

# Here are all our pyroxenes 
pxs_names = ['Clinopyroxene', 'Orthopyroxene']
pxs = df_pred[df_pred.Predict_Mineral.isin(pxs_names)]
display('Pyroxenes:', pxs)

# Here are all our oxides 
ox_names = ['Oxide']
oxs = df_pred[df_pred.Predict_Mineral.isin(ox_names)]
display('Oxides:', oxs)

Plot these feldspars, pyroxenes, and spinels! 

In [ ]:
# Use Feldsparclassifier to examine at the component space (XAn, XAb, XOr)
fspar_comp = mm.FeldsparClassifier(fspars).calculate_components()
display(fspar_comp)

# Use FeldsparClassifier to plot up these data. 
fig = mm.FeldsparClassifier(fspars).plot()

In [ ]:
# Use Feldsparclassifier to examine at the component space (En, Wo, Fs). If sodic pyroxenes are also within this input, this will plot them up in the sodic pyroxene ternary
pxs_comp = mm.PyroxeneClassifier(pxs).calculate_components()
display(pxs_comp)

# Use PyroxeneClassifier to plot up these data. 
fig = mm.PyroxeneClassifier(pxs).plot()

In [ ]:
# Use OxideClassifier to examine at the component space.
oxs_comp = mm.OxideClassifier(oxs).calculate_components()
display(oxs_comp)

# Use OxideClassifier to plot up these data. 
fig = mm.OxideClassifier(oxs).plot()

In [ ]:
np.unique(oxs_comp.Predict_Mineral)

You might note that the structure of these three ``...Classifier`` classes is identical. That is intentional! ``mm.FeldsparClassifier``, ``mm.PyroxeneClassifier``, and ``mm.OxideClassifier`` all have  `calculate_components` and `plot` methods embedded. 

### We know the mineralogy now. What if you now want to inspect the chemical variation within the individual crystals? Pull the component maps created for each sample and plot this up with ``mm.plot_component_composite``

This function currently does this calculation for feldspars, pyroxenes, and olivines as these are the most common phases I work with. This can easily be expanded with all the stoichiometric mineral functions. Here, I will just show this for these common igneous phases. 

In [ ]:
# Inspect what’s available:
print(sorted(output["component_maps"].keys()))

#  Plot map highlighting internal compositional variation
fig = mm.plot_component_composite(output, # specify output froma bove
                                  title="09g3", # optionally add a title to this plot
                                  smooth_sigma=0.25, # add a Gaussian blur to smooth compositional data, usually turned off. 
                                  scalebar_um=50, # define size of scalebar desired, in microns
                                  pixel_size_um=2, # define size of each pixel of scalebar, in microns 
                                  scalebar_loc='upper right', # specify location for scalebar
                                  scalebar_col='black', # specify color for scalebar
                                  )


One could alternatively use all the functions within mineralML.mapping to do these same things, in a more stepwise manner. Look through the documentation if you would like to use individual bits of this code. 